# Классификация текстов с помощью Decision Tree и Random Forest

## 1. Загрузка необходимых библиотек и данных

In [2]:
import re
import pandas as pd
from datasets import load_dataset

from sklearn.datasets import fetch_20newsgroups
from sklearn.feature_extraction.text import CountVectorizer, TfidfVectorizer
from sklearn.tree import DecisionTreeClassifier
from sklearn.ensemble import RandomForestClassifier
from sklearn.metrics import f1_score

import nltk
from nltk.corpus import stopwords
from nltk.stem import PorterStemmer, WordNetLemmatizer

# Загружаем необходимые ресурсы NLTK
nltk.download('punkt')
nltk.download('wordnet')
nltk.download('averaged_perceptron_tagger')
nltk.download('stopwords')

[nltk_data] Downloading package punkt to
[nltk_data]     C:\Users\alesh\AppData\Roaming\nltk_data...
[nltk_data]   Package punkt is already up-to-date!
[nltk_data] Downloading package wordnet to
[nltk_data]     C:\Users\alesh\AppData\Roaming\nltk_data...
[nltk_data]   Package wordnet is already up-to-date!
[nltk_data] Downloading package averaged_perceptron_tagger to
[nltk_data]     C:\Users\alesh\AppData\Roaming\nltk_data...
[nltk_data]   Package averaged_perceptron_tagger is already up-to-
[nltk_data]       date!
[nltk_data] Downloading package stopwords to
[nltk_data]     C:\Users\alesh\AppData\Roaming\nltk_data...
[nltk_data]   Package stopwords is already up-to-date!


True

### 1.1 Загрузка датасета emotion (train/test)

In [3]:
dataset_emotion_train = load_dataset('emotion', split='train')
dataset_emotion_test = load_dataset('emotion', split='test')

print(f"emotion train size: {len(dataset_emotion_train)}")
print(f"emotion test size: {len(dataset_emotion_test)}")

emotion train size: 16000
emotion test size: 2000


### 1.2 Загрузка датасета 20newsgroups (только указанные категории, train/test)

In [4]:
categories = ['comp.sys.ibm.pc.hardware',
              'comp.sys.mac.hardware',
              'comp.graphics',
              'comp.windows.x']

news_train = fetch_20newsgroups(subset='train', categories=categories, shuffle=True, random_state=42)
news_test = fetch_20newsgroups(subset='test', categories=categories, shuffle=True, random_state=42)

print(f"20newsgroups train size: {len(news_train.data)}")
print(f"20newsgroups test size: {len(news_test.data)}")

20newsgroups train size: 2345
20newsgroups test size: 1561


### 1.3 Функция очистки текстов для 20newsgroups (удаление метаданных, цитат, спецсимволов)

In [5]:
def clean_news_text(text):
    # Удаляем заголовки писем
    text = re.sub(r'^(From|Subject|Organization|Lines|Distribution|NNTP-Posting-Host|Keywords|Summary):.*$', '', text, flags=re.MULTILINE)
    # Удаляем цитаты (строки, начинающиеся с '>')
    text = re.sub(r'^>.*$', '', text, flags=re.MULTILINE)
    # Удаляем строки, состоящие преимущественно из повторяющихся символов (разделители)
    lines = text.split('\n')
    cleaned_lines = []
    for line in lines:
        if len(line) == 0:
            continue
        non_alnum = sum(1 for c in line if not c.isalnum() and not c.isspace())
        if non_alnum / len(line) > 0.7 and len(line) > 10:
            continue
        cleaned_lines.append(line)
    text = '\n'.join(cleaned_lines)
    # Удаляем email-адреса
    text = re.sub(r'\S+@\S+', '', text)
    # Удаляем прочие специальные символы, оставляем буквы, цифры, пробелы и базовую пунктуацию
    text = re.sub(r"[^\w\s.!?']", ' ', text)
    # Сжимаем множественные пробелы
    text = re.sub(r'\s+', ' ', text).strip()
    return text

# Применяем очистку к обучающим и тестовым текстам news
news_train_cleaned = [clean_news_text(text) for text in news_train.data]
news_test_cleaned = [clean_news_text(text) for text in news_test.data]

## 2. Предобработка текстов (лемматизация, стемминг)

In [6]:
# Инициализируем стеммер (Porter) и лемматизатор (WordNet)
stemmer = PorterStemmer()
lemmatizer = WordNetLemmatizer()
stop_words = set(stopwords.words('english'))

def tokenize_only(text):
    """Только токенизация и удаление стоп-слов (без нормализации)"""
    words = nltk.word_tokenize(text.lower())
    words = [w for w in words if w.isalpha() and w not in stop_words]
    return ' '.join(words)

def stem_text(text):
    """Стемминг + удаление стоп-слов"""
    words = nltk.word_tokenize(text.lower())
    words = [stemmer.stem(w) for w in words if w.isalpha() and w not in stop_words]
    return ' '.join(words)

def lemmatize_text(text):
    """Лемматизация + удаление стоп-слов"""
    words = nltk.word_tokenize(text.lower())
    words = [lemmatizer.lemmatize(w) for w in words if w.isalpha() and w not in stop_words]
    return ' '.join(words)

def lemmatize_noun_adj(text):
    """Лемматизация + только существительные и прилагательные + удаление стоп-слов"""
    words = nltk.word_tokenize(text.lower())
    pos_tags = nltk.pos_tag(words)
    filtered = []
    for word, tag in pos_tags:
        if word.isalpha() and word not in stop_words:
            if tag.startswith('NN') or tag.startswith('JJ'):
                filtered.append(lemmatizer.lemmatize(word))
    return ' '.join(filtered)

# Словарь методов предобработки
preprocessors = {
    'token_only': tokenize_only,
    'stemming': stem_text,
    'lemmatization': lemmatize_text,
    'lemmatization_noun_adj': lemmatize_noun_adj
}

### 2.1 Применение предобработки к данным emotion

In [7]:
emotion_train_texts = dataset_emotion_train['text']
emotion_test_texts = dataset_emotion_test['text']
emotion_train_labels = dataset_emotion_train['label']
emotion_test_labels = dataset_emotion_test['label']

processed_train = {'emotion': {}, 'news': {}}
processed_test = {'emotion': {}, 'news': {}}

for prep_name, prep_func in preprocessors.items():
    print(f"Processing emotion with {prep_name}...")
    processed_train['emotion'][prep_name] = [prep_func(text) for text in emotion_train_texts]
    processed_test['emotion'][prep_name] = [prep_func(text) for text in emotion_test_texts]

Processing emotion with token_only...
Processing emotion with stemming...
Processing emotion with lemmatization...
Processing emotion with lemmatization_noun_adj...


### 2.2 Применение предобработки к данным news (уже очищенным)

In [8]:
for prep_name, prep_func in preprocessors.items():
    print(f"Processing news with {prep_name}...")
    processed_train['news'][prep_name] = [prep_func(text) for text in news_train_cleaned]
    processed_test['news'][prep_name] = [prep_func(text) for text in news_test_cleaned]

Processing news with token_only...
Processing news with stemming...
Processing news with lemmatization...
Processing news with lemmatization_noun_adj...


## 3. Определение векторизаторов и классификаторов

In [9]:
vectorizers = {
    'binary': CountVectorizer(binary=True),
    'count': CountVectorizer(binary=False),
    'tfidf': TfidfVectorizer()
}

classifiers = {
    'Decision Tree': DecisionTreeClassifier(random_state=42),
    'Random Forest': RandomForestClassifier(random_state=42, n_jobs=-1)
}

# Список для сохранения результатов
results = []

## 4. Основной цикл экспериментов (по всем датасетам, предобработкам, векторизаторам и классификаторам)

In [10]:
for dataset_name in ['emotion', 'news']:
    print(f"\n========== ДАТАСЕТ: {dataset_name.upper()} ==========")
    y_train = emotion_train_labels if dataset_name == 'emotion' else news_train.target
    y_test = emotion_test_labels if dataset_name == 'emotion' else news_test.target

    for prep_name in preprocessors.keys():
        X_train_prep = processed_train[dataset_name][prep_name]
        X_test_prep = processed_test[dataset_name][prep_name]

        for vec_name, vectorizer in vectorizers.items():
            print(f"\nПредобработка: {prep_name}, Векторизация: {vec_name}")
            X_train_vec = vectorizer.fit_transform(X_train_prep)
            X_test_vec = vectorizer.transform(X_test_prep)

            for clf_name, clf in classifiers.items():
                clf.fit(X_train_vec, y_train)
                y_pred = clf.predict(X_test_vec)

                f1_micro = f1_score(y_test, y_pred, average='micro')
                f1_macro = f1_score(y_test, y_pred, average='macro')
                f1_weighted = f1_score(y_test, y_pred, average='weighted')

                results.append({
                    'Dataset': dataset_name,
                    'Preprocessing': prep_name,
                    'Vectorizer': vec_name,
                    'Classifier': clf_name,
                    'F1_micro': f1_micro,
                    'F1_macro': f1_macro,
                    'F1_weighted': f1_weighted
                })

                print(f"{clf_name}: micro={f1_micro:.4f}, macro={f1_macro:.4f}, weighted={f1_weighted:.4f}")


========== ДАТАСЕТ: EMOTION ==========

Предобработка: token_only, Векторизация: binary
Decision Tree: micro=0.8680, macro=0.8070, weighted=0.8687
Random Forest: micro=0.8835, macro=0.8251, weighted=0.8830

Предобработка: token_only, Векторизация: count
Decision Tree: micro=0.8675, macro=0.8039, weighted=0.8684
Random Forest: micro=0.8870, macro=0.8262, weighted=0.8867

Предобработка: token_only, Векторизация: tfidf
Decision Tree: micro=0.8630, macro=0.8062, weighted=0.8635
Random Forest: micro=0.8855, macro=0.8211, weighted=0.8838

Предобработка: stemming, Векторизация: binary
Decision Tree: micro=0.7975, macro=0.7326, weighted=0.8013
Random Forest: micro=0.8455, macro=0.7779, weighted=0.8462

Предобработка: stemming, Векторизация: count
Decision Tree: micro=0.7935, macro=0.7278, weighted=0.7984
Random Forest: micro=0.8480, macro=0.7823, weighted=0.8488

Предобработка: stemming, Векторизация: tfidf
Decision Tree: micro=0.7935, macro=0.7237, weighted=0.7954
Random Forest: micro=0.8575

## 5. Сводная таблица результатов

In [12]:
df_results = pd.DataFrame(results)
df_results = df_results.sort_values(by='F1_weighted', ascending=False)
print("Топ-20 лучших комбинаций:")
df_results

Топ-20 лучших комбинаций:


,Dataset,Preprocessing,Vectorizer,Classifier,F1_micro,F1_macro,F1_weighted
3,emotion,token_only,count,Random Forest,0.887000,0.826223,0.886735
17,emotion,lemmatization,tfidf,Random Forest,0.887500,0.832247,0.886290
13,emotion,lemmatization,binary,Random Forest,0.884000,0.821247,0.884008
5,emotion,token_only,tfidf,Random Forest,0.885500,0.821095,0.883758
1,emotion,token_only,binary,Random Forest,0.883500,0.825056,0.883021
15,emotion,lemmatization,count,Random Forest,0.881000,0.817850,0.880882
0,emotion,token_only,binary,Decision Tree,0.868000,0.807011,0.868746
2,emotion,token_only,count,Decision Tree,0.867500,0.803927,0.868418
14,emotion,lemmatization,count,Decision Tree,0.865500,0.807135,0.866445
12,emotion,lemmatization,binary,Decision Tree,0.865500,0.803871,0.866099


## Выводы

### 1. Датасеты

- emotion показал лучшие результаты, чем news:
emotion содержит более короткие текста без лишнего мусора
в news длинные технические обсуждения с шумом

### 2. Предобработка текста

- Лемматизация и простая токенизация дают схожие наивысшие результаты для emotion
- Стемминг хуже на emotion(теряем смысл при грубой обрезке)
- Фильтрация только по существительным и прилагательным на emotion ухудшают результат(глаголы важны для эмоций)
- Фильтрация только по существительным и прилагательным на news наравне с обычной лемматизацией(в новостях основную нагрузку несут сущ и прил)

### 3. Тип векторизации

- на emotion все три типа (binary, count, tf‑idf) показывают схожие результаты
- на news тоже нет ярко выраженных отклонений

### 4. Классификаторы

- Random Forest преобладает над Decision tree на обоих датасетах(второй подход более склонен к переобучкнию)

### 5. Лучшие комбинации

- emotion: lemmatization + tfidf + Random Forest
- news: lemmatization + count + Random Forest